# Sprint 3 — Colab train/dev v2
Chỉ chứa train240/dev60. Chọn GPU, upload hai ZIP; kiểm hash trước khi cài. Notebook chưa thực thi; smoke và full training là hai cổng riêng. Không mount gói test vào workspace này trước khi chọn model bằng dev.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
ROOT = Path("/content/DACN")
CODE_ZIP = Path("/content/dacn_train_dev_bundle_v4.zip")
RESOURCE_ZIP = Path("/content/dacn_phobert_resources_v4.zip")
CODE_SHA256 = 'e44d6e14b76d9c00b83b9d236c56991dd43812d043aab7635d85ffa4188e1382'
RESOURCE_SHA256 = 'eec08e9d546d155646a655a1c9c10d3386b9dd5d6c7bd1a345a825723325dc46'
ENABLE_NEURAL_TRAINING = False
ENABLE_DP_ZERO_SHOT = False
DP_RESOURCE_LOCK = None  # cleared real full FastText + checkpoint lock, not a template
MODEL = "PHOBERT-CRF"  # or PROPOSED-DYN; DP waits for a cleared active lock
CANDIDATE = "c01"
RUN_ID = "phobert_crf_colab_run001_c01"  # change for every run
BACKUP_DIRECTORY = None  # choose a persistent directory; grant Drive access yourself if used


In [ ]:
# Verify and extract the code bundle before importing its helper.
if hashlib.sha256(CODE_ZIP.read_bytes()).hexdigest() != CODE_SHA256:
    raise RuntimeError("Code ZIP checksum mismatch")
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError('Duplicate ZIP member')
    manifest = json.loads(archive.read('code_bundle_manifest.json'))
    if set(names) != set(manifest['files']) | {'code_bundle_manifest.json'}:
        raise RuntimeError('Unexpected ZIP member')
    for entry in archive.infolist():
        target = (ROOT / entry.filename).resolve()
        if "\\" in entry.filename or not target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Unsafe ZIP path")
        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError("ZIP symlink forbidden")
        if target.exists() and target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() != hashlib.sha256(archive.read(entry)).hexdigest():
            raise RuntimeError("Existing file differs; use a new workspace")
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from src.modeling.colab_handoff import check_bundle, safe_extract
from src.evaluation.dev_runner import file_hash
print(check_bundle(ROOT))
safe_extract(RESOURCE_ZIP, ROOT, RESOURCE_SHA256, "resource_bundle_manifest.json")
print("Two verified bundles extracted; test not included")


In [ ]:
import platform, shutil
print({"python": platform.python_version(), "disk_free_bytes": shutil.disk_usage(ROOT).free})
print({"os": platform.platform(), "cpu_count": os.cpu_count()})
print(Path("/proc/meminfo").read_text().splitlines()[:5])
subprocess.run(["nvidia-smi"], check=False)
if shutil.disk_usage(ROOT).free < 20 * 1024**3:
    raise RuntimeError("Need at least 20 GiB free before bootstrap/training; choose another runtime")
CACHE = ROOT / "data/interim/modeling/sprint03/colab_cache_v1"
CACHE.mkdir(parents=True, exist_ok=True)
for name in ("TMPDIR", "TEMP", "TMP", "PIP_CACHE_DIR", "UV_CACHE_DIR", "XDG_CACHE_HOME", "HF_HOME", "TORCH_HOME"):
    directory = CACHE / name.lower()
    directory.mkdir(exist_ok=True)
    os.environ[name] = str(directory)
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# The pinned profile requires an isolated Python 3.11, independent of kernel version.
UV = ROOT / ".bootstrap_uv/bin/uv"
if not UV.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(ROOT / ".bootstrap_uv"), "uv==0.8.22"], check=True)
os.environ["UV_PYTHON_INSTALL_DIR"] = str(ROOT / ".managed_python")
ENV = ROOT / ".venv_colab311"
if not (ENV / "bin/python").exists():
    subprocess.run([str(UV), "venv", "--python", "3.11", str(ENV), "--seed"], check=True)
PY = str(ENV / "bin/python")
subprocess.run([PY, "--version"], check=True)
subprocess.run([PY, "-m", "pip", "install", "torch==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([PY, "-m", "pip", "install", "-r", "configs/colab/sprint03/requirements_transitive_v1.txt"], check=True)
LOCK = ROOT / "data/interim/modeling/sprint03/task_01_06_20261003_v1/resource_lock_local_v1.json"
lock = json.loads(LOCK.read_text())
JAVA_HOME = ROOT / lock["java"]["path"]
(JAVA_HOME / "bin/java").chmod(0o755)
os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = str(JAVA_HOME / "bin") + os.pathsep + os.environ["PATH"]
os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.io.tmpdir=" + str(CACHE / "tmpdir") + " -XX:-UsePerfData"


## Resource gates
PhoBERT resources đã có trong ZIP với revision/hash/license. DP chưa có active lock: checkpoint license và full FastText còn pending. Không tự tải DP hoặc thay bằng embedding khác. RAM hệ thống cần ít nhất 10 GiB cho full FastText; GPU VRAM không thay được RAM này. Training PhoBERT cần 20 GiB đĩa trống/run và profile GPU ít nhất 8 GiB VRAM.


In [ ]:
# CPU data preparation checks the three source dependencies, without loading a model.
PREPARED = "data/interim/modeling/sprint03/colab_closure_run001"
subprocess.run([PY, "-m", "scripts.30_prepare_model_training_data", "prepare", "--output-dir", PREPARED], check=True)
# GPU preflight and 300-sample real tokenizer alignment. A new directory is required.
PREFLIGHT = "data/interim/modeling/sprint03/colab_preflight_run001"
subprocess.run([PY, "-m", "scripts.58_colab_runtime", "preflight", "--output-dir", PREFLIGHT], check=True)


In [ ]:
# One optimizer step per pretrained model on train only; checkpoints are smoke-only.
ENABLE_PRETRAINED_GPU_SMOKE = False
SMOKE = "data/interim/modeling/sprint03/colab_smoke_run001"
if ENABLE_PRETRAINED_GPU_SMOKE:
    subprocess.run([PY, "-m", "scripts.58_colab_runtime", "smoke", "--output-dir", SMOKE], check=True)
else:
    print("GPU smoke NOT_EXECUTED; full training remains blocked")


In [ ]:
# Full runs use c01 and c02 separately; smoke weights are never resumed as training.
if ENABLE_NEURAL_TRAINING:
    if BACKUP_DIRECTORY is None:
        raise RuntimeError("Configure a persistent backup outside ephemeral /content")
    from src.modeling.colab_runtime import validate_smoke_evidence
    validate_smoke_evidence(Path(SMOKE) / "smoke_report.json")
    for MODEL in ("PHOBERT-CRF", "PROPOSED-DYN"):
        for CANDIDATE in ("c01", "c02"):
            RUN_ID = "colab_run001_" + MODEL.lower().replace("-", "_") + "_" + CANDIDATE
            RUN = Path("data/processed/evaluation/sprint03") / RUN_ID
            CHECKPOINT = RUN / "checkpoints/best.pt"
            subprocess.run([PY, "-m", "scripts.58_colab_runtime", "train", "--model", MODEL,
                "--candidate", CANDIDATE, "--output-dir", str(RUN), "--smoke-evidence", SMOKE + "/smoke_report.json",
                "--backup-directory", str(BACKUP_DIRECTORY), "--enable-neural-training"], check=True)
else:
    print("Full training NOT_EXECUTED")


In [ ]:
# Only completed dev artifacts can create a final selection lock.
if ENABLE_NEURAL_TRAINING:
    for MODEL in ("PHOBERT-CRF", "PROPOSED-DYN"):
        PREFIX = "colab_run001_" + MODEL.lower().replace("-", "_")
        DEV_RUNS = ["data/processed/evaluation/sprint03/" + PREFIX + "_" + c + "_dev" for c in ("c01", "c02")]
        SELECTION = "data/interim/modeling/sprint03/" + PREFIX + "_selection"
        subprocess.run([PY, "-m", "scripts.58_colab_runtime", "select", "--model", MODEL,
            "--dev-runs", *DEV_RUNS, "--output-dir", SELECTION], check=True)
        from src.modeling.colab_runtime import backup_run
        backup_run(Path(SELECTION), BACKUP_DIRECTORY)


## Deepparse — gate riêng
DP-ZS-FT/DP-FT-FT chưa có active checkpoint license/hash lock. Không tự tải hoặc thay FastText. Sau khi nguồn đủ bằng chứng: script47 tạo DP-ZS config, scripts23/24 infer/score dev; script31 preflight → prepare → train c01/c02 → infer riêng → score dev. Native DP resume hiện là weights restart, không phải optimizer/RNG resume.

## Resume và bàn giao
PhoBERT/proposed: script58 train có `--resume-from <last.pt>` vào run mới, giữ candidate/resources/profile. best/last lưu optimizer/scheduler/RNG; backup định kỳ 300s và checksum cuối. Người chạy cấp quyền Drive nếu sử dụng; notebook không tự mount. Chỉ sau dev locks mới mở notebook final test riêng. Stability seeds chưa chạy: NOT_EXECUTED.
